# NLP Assignment 1: Kaggle build (full pipeline)

**Covers**
- Step 0: raw corpus verification, cleaned corpus, cleaning verification, vocabulary and token array.
- Q1.1 to Q1.8: n-gram model, probabilities, stopwords, sentence scores, generation (stage 10).
- Q2: held-out perplexity, seed 42 (primary) and seed 7 (replication) (stage 11).
- Q3: real versus corrupted sentence detection, with UNK baseline (stage 12).
- Q4: next-word prediction, Top-1, Top-5, MRR@5 (stage 13).
- Presentation artifacts: export bundle and checksums.

**Session settings**
- Accelerator: None (CPU). No GPU is used.
- Internet: On. The repository clone needs it.
- Input: attach the dataset that holds `GPAC.txt` or `GPAC.txt.gz` with *+ Add Input*. Attach nothing else. Do not attach a dataset with `Cleaned_GPAC.txt`, which AGENTS B2 forbids.

**Where each result is printed**
- Raw corpus hash: the "Fetch raw corpus" cell.
- Step 0 shares, Q1.3 sums, Q1.8 verbatim counts, and the Q2, Q3, Q4 results: the "Gate checks" cell.
- Q1 tables: the "Results summary" cell.

**Resuming and saving**
- Re-running the notebook resumes. Each stage skips itself if its manifest entry still matches.
- To continue after the 12-hour session cap, commit the run (*Save Version → Save & Run All*). Then open a new version, add the committed run's output as input, and run it. Cell 7 restores `data/` and `results/` from that output.
- Every number is printed by code. The markdown after each cell only describes what the printed output shows (AGENTS B8).


## 1. Runtime report


In [ ]:
import os, shutil, platform
from pathlib import Path

def read_first(path):
    try:
        with open(path) as f:
            return f.read().strip()
    except OSError:
        return None

def physical_ram_bytes():
    try:
        return os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE")
    except (ValueError, OSError):
        return None

on_kaggle = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working")
phys = physical_ram_bytes()
cgroup = read_first("/sys/fs/cgroup/memory.max") or read_first("/sys/fs/cgroup/memory/memory.limit_in_bytes")
print("Running on Kaggle             :", on_kaggle)
print("KAGGLE_KERNEL_RUN_TYPE        :", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print("Physical RAM (GiB)            :", round(phys / 2**30, 2) if phys else "unknown")
print("cgroup memory limit           :", cgroup)
print("CPUs                          :", os.cpu_count())
print("Disk free, /kaggle/working GiB:", round(shutil.disk_usage("/kaggle/working").free / 2**30, 1))
print("Python                        :", platform.python_version())
assert on_kaggle, "This is the Kaggle notebook. Use colab/Colab_Build.ipynb on Colab."


Result: the RAM and cgroup limit above are what this session enforces. The memory budget in cell 6 is computed from them.


## 2. Library versions


In [ ]:
import numpy, sklearn, pandas, pytest
print("numpy        :", numpy.__version__)
print("scikit-learn :", sklearn.__version__)
print("pandas       :", pandas.__version__)
print("pytest       :", pytest.__version__)


Result: the library versions above are recorded in `README.md` (AGENTS B4).


## 3. Clone or update the repository

The clone never deletes an existing folder, so `data/` and the manifest survive a re-run.


In [ ]:
import os, subprocess
from pathlib import Path

WORK = "/kaggle/working"
REPO = f"{WORK}/NLP_Assignment1"
REPO_URL = "https://github.com/KenaKaye/Assignment-1.git"
BRANCH = "full-corpus"

os.chdir(WORK)
if not Path(REPO, ".git").is_dir():
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(REPO)
print("Commit used by this run:")
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


Result: the commit printed above is the code version this run used. Record it in `README.md`.


## 4. Memory budget

The counting stages choose their method from this budget. It comes from the session's limit, not from a fixed number.


In [ ]:
import sys
REPO = "/kaggle/working/NLP_Assignment1"
sys.path.insert(0, REPO)
from scripts.common import memory_budget_mb, REPO_ROOT
print("Repository root :", REPO_ROOT)
print("Memory budget MB:", round(memory_budget_mb(), 0))


Result: the budget above is the limit used for every counting decision in this run.


## 5. Restore outputs from an earlier committed run, if attached

If the working copy already has a manifest, nothing is restored. Otherwise the cell looks for `NLP_Assignment1/data/manifest.json` under `/kaggle/input`. The files are copied with `copyfile` and made writable, because input files are read-only and the stages write into these folders.


In [ ]:
import glob, os, shutil
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/NLP_Assignment1")
MANIFEST = REPO_ROOT / "data" / "manifest.json"

if MANIFEST.exists():
    print("Working copy has a manifest. No restore needed.")
else:
    found = [Path(p) for p in glob.glob("/kaggle/input/**/NLP_Assignment1/data/manifest.json", recursive=True)]
    if not found:
        print("No earlier run attached under /kaggle/input. Starting with an empty data folder.")
    elif len(found) > 1:
        raise SystemExit(f"More than one earlier run is attached: {found}. Attach only one.")
    else:
        src_root = found[0].parent.parent
        print("Restoring from", src_root)
        restored = 0
        for sub in ["data", "results"]:
            for src in (src_root / sub).rglob("*"):
                if not src.is_file() or src.name == "GPAC.txt":
                    continue
                dst = REPO_ROOT / src.relative_to(src_root)
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(src, dst)
                os.chmod(dst, 0o644)
                restored += 1
        print(f"Restored {restored} files")


Result: the number of restored files is printed above. Restored files are used only if each stage's manifest still matches them, and the stages check this themselves.


## 6. Fetch the raw corpus and verify it (Step 0)

The raw file is taken from the attached dataset. Only the exact names `GPAC.txt` and `GPAC.txt.gz` are accepted. Files with "cleaned" in the name are ignored. The hash is checked against the authoritative SHA-256 in the plan.


In [ ]:
import gzip, shutil, hashlib, os
from pathlib import Path

EXPECTED = "977706859188f7598ca97a29d7203d86926c11ede591fba31915c74ae9eabcfa"
REPO_ROOT = Path("/kaggle/working/NLP_Assignment1")
RAW = REPO_ROOT / "data" / "GPAC.txt"
os.chdir(REPO_ROOT)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW.parent.mkdir(parents=True, exist_ok=True)
if RAW.exists() and sha256_of(RAW) == EXPECTED:
    print("Raw corpus already present and matching.")
else:
    matches = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        for name in files:
            if name in ("GPAC.txt", "GPAC.txt.gz") and "cleaned" not in os.path.join(root, name).lower():
                matches.append(os.path.join(root, name))
    if not matches:
        print("Contents of /kaggle/input:")
        for root, _dirs, files in os.walk("/kaggle/input"):
            print(" ", root, files)
        raise SystemExit("GPAC.txt or GPAC.txt.gz not found under /kaggle/input. Attach the dataset with + Add Input, then run this cell again.")
    if len(matches) > 1:
        raise SystemExit(f"More than one GPAC file is attached: {matches}. Attach only the dataset that holds the authoritative corpus.")
    src = matches[0]
    print("Using", src)
    if src.endswith(".gz"):
        with gzip.open(src, "rb") as s, open(RAW, "wb") as d:
            shutil.copyfileobj(s, d)
    else:
        shutil.copyfile(src, RAW)

actual = sha256_of(RAW)
print("ACTUAL   SHA-256:", actual)
print("EXPECTED SHA-256:", EXPECTED)
assert actual == EXPECTED, "Raw corpus hash mismatch. This is not the authoritative corpus. Stop here."
print("Raw corpus SHA-256 matches the authoritative value.")


Result: the two hashes printed above are identical, and the assertion passed. If the assertion failed, stop: the attached file is not the authoritative corpus.


## 7. Reset stale outputs, if any

The cleaned corpus from an older run has no sentence-final punctuation, so its stages cannot be reused. When that is detected, this cell removes stages 02 to 13. The raw file, `sentence_assessment.tsv`, the stage 01 manifest entry, and the run log are kept.


In [ ]:
import json, os, shutil
from pathlib import Path

REPO = Path("/kaggle/working/NLP_Assignment1")
os.chdir(REPO)
DATA = REPO / "data"
RESULTS = REPO / "results"
PUNCT = ("።", "?", "!")
KEEP_STAGE = "01_assess_raw"

def cleaned_is_stale(path, sample=2000, threshold=0.99):
    if not path.exists():
        print("No cleaned corpus yet. Nothing to reset.")
        return False
    total = ok = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.rstrip("\n")
            if not s:
                continue
            total += 1
            ok += s.endswith(PUNCT)
            if total >= sample:
                break
    frac = ok / total if total else 0.0
    print(f"Cleaned lines ending in sentence punctuation (first {total}): {frac:.4f}")
    return frac < threshold

if not cleaned_is_stale(DATA / "GPAC_cleaned.txt"):
    print("Cleaned corpus is current. No reset.")
else:
    print("Stale cleaned corpus. Resetting stages 02 to 13.")
    manifest_path = DATA / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
    for stage in list(manifest):
        if stage == KEEP_STAGE:
            continue
        for rel in manifest[stage].get("outputs", {}):
            p = REPO / rel
            if p.is_file():
                p.unlink()
        manifest.pop(stage)
    for pattern in ["GPAC_cleaned.txt", "*.npy", "*.tsv", "split_indices_*.json", "lut_*.npy"]:
        for p in DATA.glob(pattern):
            if p.name not in ("sentence_assessment.tsv", "GPAC.txt"):
                p.unlink()
    shutil.rmtree(DATA / "counts", ignore_errors=True)
    for p in RESULTS.glob("*"):
        if p.is_file() and p.name != "run_log.jsonl":
            p.unlink()
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    cks_path = DATA / "checksums.json"
    if cks_path.exists():
        cks = json.loads(cks_path.read_text(encoding="utf-8"))
        cks = {k: v for k, v in cks.items() if (REPO / k).exists()}
        cks_path.write_text(json.dumps(cks, indent=2, sort_keys=True), encoding="utf-8")
    print("Reset complete.")


Result: the fraction printed above is near 1 for a current cleaned corpus. If it is below 0.99, the reset ran, and the message above says so.


## 8. Run the pipeline (stages 01 to 13)

Each stage runs as a separate process and prints its own output. A failed stage stops the run and names itself. Stages with a valid manifest entry are skipped.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

REPO = "/kaggle/working/NLP_Assignment1"
os.chdir(REPO)
os.makedirs("results", exist_ok=True)

STAGES = [
    ["scripts/01_assess_raw.py"],
    ["scripts/02_build_cleaned.py"],
    ["scripts/03_verify_cleaned.py"],
    ["scripts/04_build_vocab_tokens.py"],
    ["scripts/06_split.py"],
    ["scripts/07_count_full.py"],
    ["scripts/08_count_train.py", "--seed", "42"],
    ["scripts/08_count_train.py", "--seed", "7"],
    ["scripts/10_q1_results.py"],
    ["scripts/11_q2_perplexity.py"],
    ["scripts/12_q3_extrinsic.py"],
    ["scripts/13_q4_eval.py"],
]

times_path = Path("results/kaggle_stage_times.json")
times = json.loads(times_path.read_text()) if times_path.exists() else {}

for args in STAGES:
    name = " ".join(args)
    print(f"\n==================== {name} ====================", flush=True)
    t0 = time.time()
    proc = subprocess.Popen(
        [sys.executable, "-u", *args],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    rc = proc.wait()
    dur = round(time.time() - t0, 1)
    times[name] = dur
    times_path.write_text(json.dumps(times, indent=2), encoding="utf-8")
    print(f"== {name}: exit code {rc}, elapsed {dur} s ==", flush=True)
    if rc != 0:
        raise RuntimeError(
            f"Stage failed: {name} (exit code {rc}). Fix the cause and run this cell again. "
            "Finished stages will be skipped."
        )

print("\nPipeline finished. Stage times (s):")
for k, v in times.items():
    print(f"  {k}: {v}")


Result: every stage printed its exit code and elapsed time above. Each stage's own numbers are in its output block. Stage times are saved to `results/kaggle_stage_times.json` for comparison with the plan's estimates.


## 9. Gate checks

Each check prints its value and asserts against the benchmark or the rule. Nothing is marked as passed unless its assertion ran.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import numpy as np

os.chdir("/kaggle/working/NLP_Assignment1")

def load_json(path):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Missing {path}. The stage that writes it did not complete.")
    return json.loads(p.read_text(encoding="utf-8"))

print("--- Unit tests (light suite, also run on the PC) ---")
rc = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"]).returncode
assert rc == 0, "Unit tests failed. Do not use any result from this run."
print("Unit tests exit code:", rc)

print("\n--- Step 0: cleaning and verification ---")
v = load_json("data/cleaned_verification_record.json")
print("Verification record fields:", sorted(v.keys()))
letter = v["ethiopic_letter_share_pct"]
punct = v["ethiopic_with_punctuation_share_pct"]
print(f"Ethiopic letter share     : {letter:.2f}%   (benchmark: > 80%)")
print(f"Ethiopic plus punctuation : {punct:.2f}%")
assert letter >= 80.0, "Ethiopic letter share below 80%"

print("\n--- Q1.3: counts and probability sums (AGENTS B6: within 1e-9) ---")
N = len(np.load("data/tokens.npy", mmap_mode="r"))
print(f"Token positions N: {N:,}")
for n in range(1, 5):
    c = np.load(f"data/counts/full/order{n}_counts.npy")
    total = int(c.sum())
    p_sum = total / N
    print(f"  order {n}: sum(counts) = {total:,}, sum(probabilities) = {p_sum:.12f}")
    assert total == N, f"order {n}: counts do not sum to N"
    assert abs(p_sum - 1.0) <= 1e-9, f"order {n}: probabilities do not sum to 1 within 1e-9"

print("\n--- Q1.8: verbatim copies, full corpus ---")
gen = load_json("results/q1_generation.json")
print("Keys:", list(gen.keys()))
for order_key, sents in gen.items():
    flags = [bool(s["is_full_verbatim_copy"]) for s in sents]
    print(f"  {order_key}: {sum(flags)} of {len(flags)} generated sentences are verbatim corpus lines")

print("\n--- Q2: held-out perplexity (both seeds) ---")
q2 = load_json("results/q2_perplexity.json")
print("Top-level keys:", list(q2.keys()))
print(json.dumps(q2, indent=1, ensure_ascii=False))

print("\n--- Q3: extrinsic metrics ---")
q3 = load_json("results/q3_metrics.json")
print("Top-level keys:", list(q3.keys()))
print(json.dumps(q3, indent=1, ensure_ascii=False))

print("\n--- Q4: predictive text ---")
q4 = load_json("results/q4_eval.json")
print("Top-level keys:", list(q4.keys()))
print(json.dumps(q4, indent=1, ensure_ascii=False))


Result: the unit tests, the Step 0 shares, the Q1.3 sums, the Q1.8 verbatim counts, and the full Q2, Q3, and Q4 result files are printed above. The first failed assertion stops the cell. Describe a benchmark as met only when the printed `meets_benchmarks` value is true, and a classifier as usable only when its printed `usable` value is true.


## 10. Results summary (Q1 tables)


In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 80)
for n in range(1, 5):
    print(f"\n=== Q1.3 top 10, n = {n} ===")
    print(pd.read_csv(f"results/q1_top10_n{n}.csv").to_string(index=False))

print("\n=== Q1.4 conditional probabilities ===")
print(pd.read_csv("results/q1_bigram_conditional.csv").to_string(index=False))

for n in range(1, 5):
    print(f"\n=== Q1.5 top 10 after stopword removal, n = {n} ===")
    print(pd.read_csv(f"results/q1_stopword_top10_n{n}.csv").to_string(index=False))

print("\n=== Q1.7 sentence log-probabilities ===")
print(pd.read_csv("results/q1_sentence_probs.csv").to_string(index=False))


Result: the tables above are the Q1.3, Q1.4, Q1.5, and Q1.7 outputs as the code wrote them. Q1.6 word-cloud inputs are in `results/q1_wordcloud_*.csv` and are rendered on the PC with Noto Sans Ethiopic (AGENTS B7).


## 11. Export the canonical artifacts

The list comes from `scripts/artifacts.py`. The export stops if any listed artifact is missing. The zip is created after every file is copied, so it includes the checksum, manifest, and log files.


In [ ]:
import json, os, shutil, sys
from pathlib import Path

REPO = Path("/kaggle/working/NLP_Assignment1")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from scripts.artifacts import write_artifact_list

write_artifact_list()
artifacts = json.loads(Path("data/artifact_list.json").read_text(encoding="utf-8"))

EXPORT = Path("/kaggle/working/export")
if EXPORT.exists():
    shutil.rmtree(EXPORT)
EXPORT.mkdir(parents=True)

missing, total = [], 0
for rel in artifacts:
    src = REPO / rel
    if not src.is_file():
        missing.append(rel)
        continue
    dst = EXPORT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    total += dst.stat().st_size

for extra in ["data/checksums.json", "data/manifest.json", "data/artifact_list.json",
              "results/run_log.jsonl", "results/kaggle_stage_times.json"]:
    src = REPO / extra
    if src.is_file():
        dst = EXPORT / extra
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        total += dst.stat().st_size

print(f"Listed artifacts : {len(artifacts)}")
print(f"Copied           : {len(artifacts) - len(missing)}")
print(f"Missing          : {missing}")
print(f"Total size       : {total / 2**30:.2f} GiB")
assert not missing, "Canonical artifacts are missing. Fix them before exporting."

zip_path = shutil.make_archive("/kaggle/working/nlp_assignment1_artifacts", "zip", EXPORT)
shutil.rmtree(EXPORT)
print("Created", zip_path, round(os.path.getsize(zip_path) / 2**30, 2), "GiB")
print("Removed the unzipped export folder so it is not stored twice.")


Result: the listed, copied, and missing counts, the total size, and the zip size are printed above. The zip is the download for your PC.


## 12. Output size and cleanup

Kaggle stores everything under `/kaggle/working` as the notebook's output. The raw corpus is about 1 GB and can be recopied from the attached dataset on the next run, so it is removed from the output by default. Set `DROP_RAW_FROM_OUTPUT = False` to keep it.


In [ ]:
import os
from pathlib import Path

DROP_RAW_FROM_OUTPUT = True
RAW = Path("/kaggle/working/NLP_Assignment1/data/GPAC.txt")

if DROP_RAW_FROM_OUTPUT and RAW.exists():
    RAW.unlink()
    print("Removed data/GPAC.txt from the output. Cell 6 recopies it from the attached dataset on the next run.")

def folder_bytes(root):
    return sum(p.stat().st_size for p in Path(root).rglob("*") if p.is_file())

out_bytes = folder_bytes("/kaggle/working")
print(f"Output folder size: {out_bytes / 2**30:.2f} GiB")
print("Zip:", Path("/kaggle/working/nlp_assignment1_artifacts.zip").stat().st_size / 2**30, "GiB")
print("Check this size against the output limit shown in the Kaggle session before committing.")


Result: the output size is printed above. If Kaggle rejects the commit for size, set `DROP_RAW_FROM_OUTPUT` as above and check the size again. The zip alone is the minimum deliverable.


## 13. Next steps on the PC

1. Download `nlp_assignment1_artifacts.zip` from the notebook's **Output** tab.
2. Extract it into the repository root. The `data/` and `results/` folders merge with the existing ones.
3. Run `python scripts/check_download.py` (or `uv run scripts/check_download.py`). Every checksum and the raw corpus hash must match. Do not run anything else until this passes (AGENTS B13).
4. Then run `python -m pytest tests/`, and launch the CLI with `python app/cli.py`.
